# Desarrollo de Aplicaciones Web (DAB-2201)
## Sesión 02: Gestión en GitHub Web y Métodos HTTP (GET, POST, PUT, DELETE)
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-desarrollo-web`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Enrutamiento manual por verbo HTTP*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Especificación semántica de verbos e idempotencia* y formular prompt.
3. **Fase 3 (El Auditor):** Someter el servidor al Test Harness: *Pruebas de métodos no permitidos (HTTP 405)*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion02.md`.

### Paso 0: Inicialización del Entorno (FastAPI y PHP 8+)

In [23]:
import os
print('⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...')
!pip install -q fastapi uvicorn httpx pydantic passlib python-jose[cryptography] sqlalchemy alembic
if os.system('which php > /dev/null 2>&1') != 0:
    print('⏳ Instalando PHP CLI...')
    !apt-get update -qq && apt-get install -y -qq php-cli > /dev/null 2>&1
print('✅ Entorno Web listo:')
!python3 --version
!php -v | head -n 1

⏳ Instalando FastAPI, Uvicorn y HTTPX para pruebas...
✅ Entorno Web listo:
Python 3.13.16
PHP 8.3.6 (cli) (built: Oct 19 2026 15:16:58) (NTS)


---
## SECCIÓN A: TRACK PYTHON / FASTAPI
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Enrutamiento manual por verbo HTTP

In [24]:
%%writefile app_artesano.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Artesano")

class PayloadRequest(BaseModel):
    dato: str

# TODO (El Artesano): Implementar lógica manual del endpoint
@app.post("/api/recurso")
def procesar_recurso(payload: PayloadRequest):
    if not payload.dato.strip():
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Dato vacio")
    return {"status": "success", "mensaje": f"Procesado: {payload.dato}"}


Overwriting app_artesano.py


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [25]:

%%writefile app_ia.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

app = FastAPI(title="API Optimizada con IA")


class PayloadRequest(BaseModel):
    dato: str


recurso = None


# Consultar un recurso
@app.get("/api/recurso")
def consultar_recurso():
    if recurso is None:
        raise HTTPException(
            status_code=404,
            detail="Recurso no encontrado"
        )
    return recurso


# Crear un recurso
@app.post("/api/recurso", status_code=status.HTTP_201_CREATED)
def crear_recurso(payload: PayloadRequest):
    global recurso

    if not payload.dato.strip():
        raise HTTPException(
            status_code=400,
            detail="Dato vacío"
        )

    if recurso is not None:
        raise HTTPException(
            status_code=409,
            detail="El recurso ya existe"
        )

    recurso = {"dato": payload.dato}
    return {
        "status": "success",
        "mensaje": f"Creado: {payload.dato}"
    }


# Actualizar un recurso
@app.put("/api/recurso")
def actualizar_recurso(payload: PayloadRequest):
    global recurso

    if not payload.dato.strip():
        raise HTTPException(
            status_code=400,
            detail="Dato vacío"
        )

    recurso = {"dato": payload.dato}
    return {
        "status": "success",
        "mensaje": f"Actualizado: {payload.dato}"
    }


# Eliminar un recurso
@app.delete("/api/recurso")
def eliminar_recurso():
    global recurso
    recurso = None

    return {
        "status": "success",
        "mensaje": "Recurso eliminado"
    }

# Pega aquí el código generado por la IA para auditarlo


Overwriting app_ia.py


### Fase 3: El Auditor (Test Harness HTTP)
Pruebas de servidor: Pruebas de métodos no permitidos (HTTP 405)

In [1]:
import os
# Liberamos el puerto por si quedó un servidor colgado de la prueba anterior
os.system("fuser -k 8000/tcp > /dev/null 2>&1")

import threading
import time
import uvicorn
import httpx
from app_ia import app  # <--- Importante: auditamos la IA, no el artesano

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

# Levantar servidor en un hilo
threading.Thread(target=run_server, daemon=True).start()
time.sleep(1)

client = httpx.Client(base_url="http://127.0.0.1:8000")
print("=== AUDITORIA CRITICA WEB: SESION 02 ===\n")

# Prueba 1: POST correcto (201 Created)
print("-> Prueba 1: Creando recurso (POST)...")
r1 = client.post("/api/recurso", json={"dato": "Prueba 1"})
print(f"Código: {r1.status_code}")
print(f"Respuesta: {r1.json()}\n")

# Prueba 2: El reto de la sesión, método incorrecto (405 Method Not Allowed)
print("-> Prueba 2: Probando método no permitido...")
# Mandamos un PATCH, que no está definido en la API
r2 = client.patch("/api/recurso", json={"dato": "Prueba 2"})
print(f"Código: {r2.status_code}")
print(f"Respuesta: {r2.json()}\n")

# Validación final
if r1.status_code == 201 and r2.status_code == 405:
    print("✅ RESULTADO: Auditoría superada. Idempotencia y métodos HTTP correctos.")
else:
    print("❌ RESULTADO: Falló la auditoría. Revisar códigos de estado.")

=== AUDITORIA CRITICA WEB: SESION 02 ===

-> Prueba 1: Creando recurso (POST)...
Código: 201
Respuesta: {'status': 'success', 'mensaje': 'Creado: Prueba 1'}

-> Prueba 2: Probando método no permitido...
Código: 405
Respuesta: {'detail': 'Method Not Allowed'}

✅ RESULTADO: Auditoría superada. Idempotencia y métodos HTTP correctos.
